# DSC2026 — Prism-2B private inference on Kaggle 2×T4

Frozen contract from local calibration:

- base: `infgrad/Prism-Qwen3.5-Reranker-2B`
- LoRA alpha: **32**
- passage aggregation: **top2_max**
- max length: **1024**
- score: `logit(yes) - logit(no)`

Attach a Kaggle Dataset containing:
- `best_state.pt`
- `PRISM_PRIVATE_PASSAGES.pkl`

Enable **GPU T4 x2** and Internet.


In [8]:
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "transformers", "torchao"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir",
    "transformers==5.17.0",
    "peft==0.21.0",
    "accelerate",
    "safetensors",
    "sentencepiece",
])

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 232.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 221.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 356.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 336.5 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1


0

In [9]:
import sys
import transformers
import peft

print("Python:", sys.executable)
print("Transformers:", transformers.__version__)
print("Transformers path:", transformers.__file__)
print("PEFT:", peft.__version__)

from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM

print("Qwen3.5 import: PASS")

Python: /usr/bin/python3
Transformers: 5.17.0
Transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
PEFT: 0.21.0
Qwen3.5 import: PASS


In [10]:
import subprocess, sys

print(subprocess.check_output([
    sys.executable,
    "-c",
    "import transformers; "
    "print(transformers.__version__); "
    "from transformers import Qwen3_5TextConfig; "
    "print('Qwen3.5 subprocess PASS')"
], text=True))

5.17.0
Qwen3.5 subprocess PASS



In [11]:
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-build-isolation",
    "causal-conv1d"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "flash-linear-attention==0.5.2"
])

  Using cached causal_conv1d-1.7.0.tar.gz (30 kB)
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for causal-conv1d: filename=causal_conv1d-1.7.0-cp312-cp312-linux_x86_64.whl size=196558813 sha256=8e81f8c76435ad31aa41edc6c0c9d26de971a293b190183d2816da71547614d7
  Stored in directory: /root/.cache/pip/wheels/6f/0b/1b/e063b1e314a5ff0783e450525921cb8f5c48fbb03639bad6ba
Successfully built causal-conv1d
  Using cached flash_linear_attention-0.5.2-py3-none-any.whl.metadata (45 kB)
  Using cached fla_core-0.5.2-py3-none-any.whl.metadata (45 kB)
Using cached flash_linear_attention-0.5.2-py3-none-any.whl (399 kB)
Using cached fla_core-0.5.2-py3-none-any.whl (819 kB)


0

In [3]:
import os, sys, subprocess, importlib.util, torch
print("torch", torch.__version__)
print("GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, f"{p.total_memory/2**30:.1f} GB")
assert torch.cuda.device_count() >= 2, "Select Kaggle accelerator: GPU T4 x2"
need = [p for p in ("transformers","peft","accelerate") if importlib.util.find_spec(p) is None]
if need:
    subprocess.check_call([sys.executable,"-m","pip","install","-q"]+need)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")


torch 2.10.0+cu128
GPUs 2
0 Tesla T4 14.6 GB
1 Tesla T4 14.6 GB


'expandable_segments:True'

In [4]:
from pathlib import Path
INPUT = Path("/kaggle/input")
ck = list(INPUT.rglob("best_state.pt"))
wl = list(INPUT.rglob("PRISM_PRIVATE_PASSAGES.pkl"))
print("checkpoint", ck)
print("workload", wl)
assert len(ck)==1 and len(wl)==1
CHECKPOINT, WORKLOAD = ck[0], wl[0]
OUT = Path("/kaggle/working/prism_private")
OUT.mkdir(parents=True, exist_ok=True)


checkpoint [PosixPath('/kaggle/input/datasets/hoangnguyenkaggle62/endgame/best_state.pt')]
workload [PosixPath('/kaggle/input/datasets/hoangnguyenkaggle62/endgame/PRISM_PRIVATE_PASSAGES.pkl')]


In [5]:
from pathlib import Path
WORKER = 'import argparse, os, pickle, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nBASE_MODEL = "infgrad/Prism-Qwen3.5-Reranker-2B"\nSYSTEM_PROMPT = "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "\nINSTRUCTION = (\n    \'Judge if the document is relevant to the query. Reply "yes" or "no".\\n\'\n    \'On "yes", also emit:\\n\'\n    "<contribution>One sentence covering every core point the document contributes to the query, without elaboration.</contribution>\\n"\n    "<evidence>Self-contained rewrite of the query-relevant content. Rules:\\n"\n    "- Faithful: rephrase only; add or infer nothing.\\n"\n    "- Self-contained: evidence alone must fully answer the query.\\n"\n    "- Concise: drop query-irrelevant background.\\n"\n    "- Verbatim (no translation): proper nouns, terms, abbreviations, numbers, dates, code, URLs.\\n"\n    "- Output language: multilingual doc → query\'s language; else doc\'s language.</evidence>"\n)\nTEMPLATE = (\n    "<|im_start|>system\\n{system}<|im_end|>\\n"\n    "<|im_start|>user\\n<Instruct>: {instruction}\\n<Query>: {query}\\n<Document>: {doc}<|im_end|>\\n"\n    "<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n)\n\ndef mkprompt(q,d):\n    return TEMPLATE.format(system=SYSTEM_PROMPT, instruction=INSTRUCTION, query=q, doc=d)\n\ndef atomic_pickle(path, obj):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_bytes(pickle.dumps(obj, protocol=5))\n    os.replace(tmp, path)\n\ndef infer_contract(state):\n    a = [k for k in state if ".lora_A." in k]\n    ranks = {int(state[k].shape[0]) for k in a}\n    if len(ranks) != 1:\n        raise RuntimeError(ranks)\n    rank = next(iter(ranks))\n    targets = sorted({k.split(".lora_A.")[0].split(".")[-1] for k in a})\n    return rank, targets\n\ndef load_model(checkpoint, gpu):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from peft import LoraConfig, TaskType, get_peft_model\n    torch.cuda.set_device(gpu)\n    device = f"cuda:{gpu}"\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    state = obj.get("state_dict", obj)\n    rank, targets = infer_contract(state)\n    tok = AutoTokenizer.from_pretrained(BASE_MODEL)\n    tok.padding_side = "left"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    base = AutoModelForCausalLM.from_pretrained(\n        BASE_MODEL,\n        torch_dtype=torch.float16,\n        low_cpu_mem_usage=True,\n        attn_implementation="sdpa",\n    )\n    cfg = LoraConfig(\n        r=rank, lora_alpha=32, target_modules=targets, lora_dropout=0.0,\n        bias="none", task_type=TaskType.CAUSAL_LM,\n    )\n    model = get_peft_model(base, cfg)\n    bad = model.load_state_dict(state, strict=False)\n    if bad.unexpected_keys:\n        raise RuntimeError(bad.unexpected_keys[:10])\n    model.eval().to(device)\n    yes = tok.encode("yes", add_special_tokens=False)\n    no = tok.encode("no", add_special_tokens=False)\n    if len(yes) != 1 or len(no) != 1:\n        raise RuntimeError((yes,no))\n    return model, tok, yes[0], no[0], device\n\n@torch.inference_mode()\ndef score_batch(model, tok, yes_id, no_id, device, texts, max_length):\n    enc = tok(\n        texts, padding=True, truncation=True, max_length=max_length,\n        return_tensors="pt", add_special_tokens=False\n    )\n    enc = {k:v.to(device, non_blocking=True) for k,v in enc.items()}\n    base = model.get_base_model()\n    with torch.autocast(device_type="cuda", dtype=torch.float16):\n        out = base.model(\n            input_ids=enc["input_ids"],\n            attention_mask=enc.get("attention_mask"),\n            use_cache=False,\n            return_dict=True,\n        )\n        h = out.last_hidden_state[:, -1, :]\n        logits = base.lm_head(h).float()\n        s = logits[:, yes_id] - logits[:, no_id]\n    return s.detach().cpu().tolist()\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--gpu", type=int, required=True)\n    ap.add_argument("--shard", type=int, required=True)\n    ap.add_argument("--num-shards", type=int, required=True)\n    ap.add_argument("--workload", required=True)\n    ap.add_argument("--checkpoint", required=True)\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--batch-size", type=int, default=12)\n    ap.add_argument("--max-length", type=int, default=1024)\n    ap.add_argument("--save-every", type=int, default=25)\n    args = ap.parse_args()\n\n    model, tok, yes_id, no_id, device = load_model(args.checkpoint, args.gpu)\n    obj = pickle.loads(Path(args.workload).read_bytes())\n    queries = obj["queries"]\n    all_qids = sorted(queries)\n    qids = [q for i,q in enumerate(all_qids) if i % args.num_shards == args.shard]\n\n    out = Path(args.output)\n    saved = pickle.loads(out.read_bytes()) if out.is_file() else {}\n    remain = [q for q in qids if q not in saved or len(saved[q]) != len(queries[q]["docs"])]\n\n    bs = args.batch_size\n    start = time.perf_counter()\n    for qi,q in enumerate(remain,1):\n        row = queries[q]\n        owners, prompts = [], []\n        for d, passages in row["docs"].items():\n            for p in passages:\n                owners.append(str(d))\n                prompts.append(mkprompt(row["question"], p))\n        vals = []\n        pos = 0\n        while pos < len(prompts):\n            chunk = prompts[pos:pos+bs]\n            try:\n                vals.extend(score_batch(model,tok,yes_id,no_id,device,chunk,args.max_length))\n                pos += len(chunk)\n            except torch.cuda.OutOfMemoryError:\n                torch.cuda.empty_cache()\n                if bs <= 1:\n                    raise\n                bs = max(1, bs//2)\n                print(f"[gpu{args.gpu}] OOM -> batch={bs}", flush=True)\n\n        scores = {str(d):-1e30 for d in row["docs"]}\n        for d,s in zip(owners, vals):\n            if not np.isfinite(s):\n                raise RuntimeError((q,d,s))\n            scores[d] = max(scores[d], float(s))\n        saved[q] = scores\n\n        if qi % args.save_every == 0 or qi == len(remain):\n            atomic_pickle(out, saved)\n            elapsed = time.perf_counter()-start\n            rate = elapsed/max(qi,1)\n            print(\n                f"[gpu{args.gpu}] {qi}/{len(remain)} new | saved={len(saved)}/{len(qids)} | "\n                f"{rate:.2f}s/q | ETA={rate*(len(remain)-qi)/60:.1f}m | batch={bs}",\n                flush=True\n            )\n    print(f"[gpu{args.gpu}] COMPLETE", flush=True)\n\nif __name__ == "__main__":\n    main()\n'
wp = Path('/kaggle/working/prism_worker.py')
wp.write_text(WORKER, encoding='utf-8')
print(wp)


/kaggle/working/prism_worker.py


## Launch both T4 workers

In [12]:
import subprocess, os, sys
env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
procs, handles = [], []
for gpu in (0,1):
    log = OUT / f"worker_gpu{gpu}.log"
    h = open(log, "w", buffering=1)
    handles.append(h)
    cmd = [
        sys.executable, "/kaggle/working/prism_worker.py",
        "--gpu", str(gpu),
        "--shard", str(gpu),
        "--num-shards", "2",
        "--workload", str(WORKLOAD),
        "--checkpoint", str(CHECKPOINT),
        "--output", str(OUT / f"prism_scores_shard{gpu}.pkl"),
        "--batch-size", "32",
        "--max-length", "1024",
        "--save-every", "25",
    ]
    p = subprocess.Popen(cmd, stdout=h, stderr=subprocess.STDOUT, env=env)
    procs.append(p)
    print("launched gpu", gpu, "pid", p.pid)


launched gpu 0 pid 289
launched gpu 1 pid 290


## Monitor progress — re-run this cell any time

In [13]:
import time
for _ in range(20):
    print("="*100)
    alive = False
    for gpu,p in enumerate(procs):
        log = OUT / f"worker_gpu{gpu}.log"
        lines = log.read_text(encoding="utf-8", errors="replace").splitlines()[-6:] if log.exists() else []
        print(f"GPU {gpu} return={p.poll()}")
        print("\n".join(lines))
        alive |= (p.poll() is None)
    if not alive:
        break
    time.sleep(30)


GPU 0 return=None

GPU 1 return=None

GPU 0 return=None
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
GPU 1 return=None
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
GPU 0 return=None

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 92.48it/s] 
GPU 1 return=None
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 86.58it/s] 
GPU 0 return=None

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 92.48it/s] 
GPU 1 return=None
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 86.58it/s] 
GPU 0 return=None

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 92.48it/s] 
GPU 1 return=None
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 86.58it/s] 
GPU 0 return=None

Loading weights: 100%|██████████| 320/320 [00:03<00:00, 92.4

KeyboardInterrupt: 

## Wait for completion

In [ ]:
codes = [p.wait() for p in procs]
for h in handles:
    h.close()
print("return codes", codes)
for gpu in (0,1):
    log = OUT / f"worker_gpu{gpu}.log"
    print("="*100)
    print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))
assert codes == [0,0]


## Merge and validate

In [ ]:
import pickle, hashlib, json
obj = pickle.loads(WORKLOAD.read_bytes())
queries = obj["queries"]
merged = {}
for gpu in (0,1):
    part = pickle.loads((OUT/f"prism_scores_shard{gpu}.pkl").read_bytes())
    assert not (set(merged) & set(part))
    merged.update(part)
assert len(merged)==2080

pairs = 0
missing = []
for q,row in queries.items():
    pairs += len(row["docs"])
    if q not in merged:
        missing.append((q,None))
        continue
    for d in row["docs"]:
        if d not in merged[q]:
            missing.append((q,d))
assert not missing, missing[:20]

final = Path("/kaggle/working/prism_private_scores.pkl")
final.write_bytes(pickle.dumps(merged, protocol=5))
sha = hashlib.sha256(final.read_bytes()).hexdigest()
vals = [s for row in merged.values() for s in row.values()]
report = {
    "queries": len(merged),
    "pairs": pairs,
    "score_min": float(min(vals)),
    "score_max": float(max(vals)),
    "sha256": sha,
    "base": "infgrad/Prism-Qwen3.5-Reranker-2B",
    "alpha": 32,
    "mode": "top2_max",
    "max_length": 1024,
}
Path("/kaggle/working/PRISM_KAGGLE_REPORT.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("DOWNLOAD:", final)
